In [1]:
import axelrod as axl
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
from axelrod.action import Action, actions_to_str
from axelrod.player import Player
from axelrod.strategy_transformers import (
    FinalTransformer,
    TrackHistoryTransformer,
)
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import numpy as np
from collections import Counter
import pandas as pd
from math import exp
import numpy as np
from functools import reduce

In [2]:
import os


p = [player() for player in axl.filtered_strategies({
        "long_run_time": False,
})]

fuzzy_players = [
    axl.FuzzyMamdaniRules8(),
    axl.FuzzyMamdaniRules7(),
    axl.FuzzyMamdaniRules6(),
    axl.FuzzyMamdaniRules5(),
]

for fp in fuzzy_players:
    p.append(fp)

    dfs = []

    with pd.ExcelWriter(f"Axelrod-{fp.name}.xlsx") as writer:
        print("Kreiran novi fajl")


    for i in range(0, 30):
        print(f"Krećem {i+1}. turnir")
        t = axl.Tournament(players=p, repetitions=1)

        results = t.play()

        results.scores

    
        df = pd.DataFrame(
                {
                    "Name": [x.name for x in p],
                    "#Wins": list(map(lambda x: sum(x), results.wins)),
                    "TotalScore": list(map(lambda x: sum(x), results.scores)),
                    "CooperationRating" : results.cooperating_rating,
                    "GoodPartnerRating" : results.good_partner_rating,
                    "EigenJesusRating" : results.eigenjesus_rating,
                    "EigenMosesRating" : results.eigenmoses_rating,
                })


        df = df.sort_values("TotalScore", ascending=False)
        df = df.reset_index(drop=True)
        dfs.append(df)

    i = 1
    
    for df in dfs:
        with pd.ExcelWriter(f"Axelrod-{fp.name}.xlsx", mode="a" if os.path.exists(f"Axelrod-{fp.name}.xlsx") else "w", engine="openpyxl") as writer:
            df.to_excel(writer, sheet_name="Rank_List_"+str(i), startrow=0)
            i = i+1
    
    prepared = [
    df[["Name", "TotalScore"]].rename(
        columns={"TotalScore": f"TotalScore{i+1}"}
    )
    for i, df in enumerate(dfs)
    ]

    # Merge svih DF-ova po Name
    result_df = reduce(
        lambda left, right: pd.merge(left, right, on="Name", how="outer"),
        prepared
    )

    with pd.ExcelWriter(f"Axelrod-{fp.name}.xlsx", mode="a", engine="openpyxl") as writer:
            result_df.to_excel(writer, sheet_name="Total Score", startrow=0)

    combined = pd.concat(dfs)
        
    agg = combined.groupby('Name').agg({
            '#Wins': 'sum',
            'TotalScore': 'sum',
            'CooperationRating': ['mean', 'std'],
            'GoodPartnerRating': ['mean', 'std'],
            'EigenJesusRating': ['mean', 'std'],
            'EigenMosesRating': ['mean', 'std']
    }).reset_index()

    agg.columns = ['Name', '#Wins', 'TotalScore',
                'AvgCoopRating', 'SD_CoopRating',
                'AvgGP', 'SD_GP',
                'AvgEJ', 'SD_EJ',
                'AvgEM', 'SD_EM']
            
            # Compute %Wins and AvgScore
    num_dfs = len(dfs)
    total_matches = (len(p)-1) * num_dfs
    agg['%Wins'] = agg['#Wins'] / total_matches
    agg['AvgScore'] = agg['TotalScore'] / total_matches
            
            # Keep only desired columns
    result = agg[['Name', '%Wins', 'AvgScore',
                'AvgCoopRating', 'SD_CoopRating',
                'AvgGP', 'SD_GP',
                'AvgEJ', 'SD_EJ',
                'AvgEM', 'SD_EM']]
    result = result.sort_values(by="AvgScore", ascending=False)


    with pd.ExcelWriter(f"Axelrod-{fp.name}.xlsx", mode="a", engine="openpyxl") as writer:
            result.to_excel(writer, sheet_name="FINAL", startrow=0)

    # Kreiramo prazan dict gde će ključevi biti imena, a vrednosti liste rangova
    rank_dict = {}

    for i, df in enumerate(dfs):
        # Rang po TotalScore u toj iteraciji
        ranks = df.set_index('Name')['TotalScore'].rank(ascending=False, method='min')
        
        for name, rank in ranks.items():
            if name not in rank_dict:
                rank_dict[name] = []
            rank_dict[name].append(rank)

    # Pretvaramo u DataFrame
    rank_df = pd.DataFrame.from_dict(rank_dict, orient='index')

    # Preimenuj kolone u Rank_1, Rank_2, ...
    rank_df.columns = [f'Rank_{i+1}' for i in range(len(rank_df.columns))]

    # Reset index da Name bude kolona
    rank_df = rank_df.reset_index().rename(columns={'index': 'Name'})

    # rank_df sada sadrži 30 kolona rangova po iteracijama
    with pd.ExcelWriter(f"Axelrod-{fp.name}.xlsx", mode="a", engine="openpyxl") as writer:
        rank_df.to_excel(writer, sheet_name="Ranks", startrow=0)

    p.remove(fp)

Kreiran novi fajl
Krećem 1. turnir


Analysing: 100%|██████████| 25/25 [00:22<00:00,  1.12it/s]


Krećem 2. turnir


Analysing: 100%|██████████| 25/25 [00:16<00:00,  1.54it/s]


Krećem 3. turnir


Analysing: 100%|██████████| 25/25 [00:15<00:00,  1.62it/s]


... (670 outputs left out)

Error computing output for FuzzyMamdaniRules5
Inputs: cooperation=99.0, adaptivity=0, forgiveness=100, stochastic=4.123711340206185
Error computing output for FuzzyMamdaniRules5
Inputs: cooperation=99.00990099009901, adaptivity=0, forgiveness=100, stochastic=4.081632653061225
Error computing output for FuzzyMamdaniRules5
Inputs: cooperation=99.01960784313727, adaptivity=0, forgiveness=100, stochastic=4.040404040404041
Error computing output for FuzzyMamdaniRules5
Inputs: cooperation=99.02912621359224, adaptivity=0, forgiveness=100, stochastic=4.0
Error computing output for FuzzyMamdaniRules5
Inputs: cooperation=99.03846153846155, adaptivity=0, forgiveness=100, stochastic=3.9603960396039604
Error computing output for FuzzyMamdaniRules5
Inputs: cooperation=99.04761904761905, adaptivity=0, forgiveness=100, stochastic=3.9215686274509802
Error computing output for FuzzyMamdaniRules5
Inputs: cooperation=99.05660377358491, adaptivity=0, forgiveness=100, stochastic=3.8834951456310676
Error com

Playing matches:  90%|████████▉ | 23183/25878 [03:30<00:53, 49.99it/s] 

Error computing output for FuzzyMamdaniRules5
Inputs: cooperation=99.39024390243902, adaptivity=0, forgiveness=100, stochastic=2.484472049689441
Error computing output for FuzzyMamdaniRules5
Inputs: cooperation=99.39393939393939, adaptivity=0, forgiveness=100, stochastic=2.4691358024691357
Error computing output for FuzzyMamdaniRules5
Inputs: cooperation=99.3975903614458, adaptivity=0, forgiveness=100, stochastic=2.4539877300613497
Error computing output for FuzzyMamdaniRules5
Inputs: cooperation=99.40119760479041, adaptivity=0, forgiveness=100, stochastic=2.4390243902439024
Error computing output for FuzzyMamdaniRules5
Inputs: cooperation=99.40476190476191, adaptivity=0, forgiveness=100, stochastic=2.4242424242424243
Error computing output for FuzzyMamdaniRules5
Inputs: cooperation=99.40828402366864, adaptivity=0, forgiveness=100, stochastic=2.4096385542168677
Error computing output for FuzzyMamdaniRules5
Inputs: cooperation=99.41176470588235, adaptivity=0, forgiveness=100, stochastic

Analysing: 100%|██████████| 25/25 [00:11<00:00,  2.10it/s]
